In [1]:
import numpy as np
import pandas as pd
import re, sklearn

In [2]:
from nltk.corpus import movie_reviews

data = [ (movie_reviews.raw(fileid), movie_reviews.categories(fileid)[0]) for fileid in movie_reviews.fileids() ]
df = pd.DataFrame(data[:100] + data[-100:], columns = ['text', 'category'])
df.head()

,text,category
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard's quick movie review \ndamn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' firs...",neg
4,synopsis : a mentally unstable man undergoing ...,neg


In [3]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

def lower(text):
    return text.lower()

def remove_punct(text):
    text = re.sub(r'[^\w\s+]', '', text)
    text = re.sub(r'https?://\s+', '', text)
    return text

def tokenize(text):
    return word_tokenize(text)

def remove_stopwords(tokens):
    stop_words = set(stopwords.words('english'))
    return [ token for token in tokens if token not in stop_words and token.isalpha()]

def stemming(tokens):
    stemmer = PorterStemmer()
    return [ stemmer.stem(token) for token in tokens ]

def preprocessing(text):
    text = lower(text)
    text = remove_punct(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = stemming(tokens)
    return ' '.join(tokens)

preprocessing("hi how    are you ?  https://google.com . Its working or not , Vishal is a basketball player")

'hi httpsgooglecom work vishal basketbal player'

In [4]:
# Train Test Split
from sklearn.model_selection import train_test_split

train_x, test_x, train_y, test_y = train_test_split(df['text'], df['category'], test_size=0.2)
train_x.shape, train_y.shape, test_x.shape, test_y.shape

((160,), (160,), (40,), (40,))

In [5]:
# Preprocessing Layer and Category Encoding
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
train_x = train_x.apply(preprocessing)
train_y = encoder.fit_transform(train_y)

In [6]:
# Text Encoding
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
train_x = vectorizer.fit_transform(train_x)

vocab_size = train_x.shape[1]
embedding_dim = 64
train_x, train_y

(<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 40611 stored elements and shape (160, 9367)>,
 array([1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1,
        1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0,
        0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0,
        0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1,
        0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0,
        0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
        0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1,
        1, 1, 0, 0, 1, 0]))

In [7]:
# Model Creation

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, SimpleRNN, Embedding

model = Sequential([
    Embedding(input_dim = vocab_size, output_dim = embedding_dim, input_length = vocab_size),
    SimpleRNN(512, activation='relu'),
    Dense(512, activation='relu'),
    Dense(128, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

C:\Users\kgliv\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer = Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics = ['accuracy']
)

history = model.fit(train_x, train_y)

5/5 ━━━━━━━━━━━━━━━━━━━━ 191s 37s/step - accuracy: 0.4688 - loss: 0.7003


In [10]:
from sklearn.metrics import classification_report

def predict(test_x, test_y):
    test_x = test_x.apply(preprocessing)
    test_x = vectorizer.transform(test_x)
    test_y = encoder.transform(test_y)

    predictions = model.predict(test_x)
    predictions = np.where(predictions < 0.5, 0, 1)

    return predictions.flatten(), test_y

predictions, actual_y = predict(test_x, test_y)
print(classification_report(actual_y, predictions))

2/2 ━━━━━━━━━━━━━━━━━━━━ 20s 5s/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        19
           1       0.53      1.00      0.69        21

    accuracy                           0.53        40
   macro avg       0.26      0.50      0.34        40
weighted avg       0.28      0.53      0.36        40



C:\Users\kgliv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\kgliv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\kgliv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo